In [1]:
import numpy as np
import pandas as pd
import data_loader as dl
df_monthly = dl.df_monthly_updated.copy()


In [2]:
df_monthly


,Unnamed: 0,ID,YEAR,MONTH,PRCP,TMAX,TMIN,TAVG,spei1,spei3,spei6,spei12,spei24,DATE,W_YEAR
0,4682,USC00011694,1926,1,260.1,11.912903,0.803226,6.358065,1.856108,2.041880,NaN,NaN,NaN,1926-01-01,1926
1,4683,USC00011694,1927,1,16.3,15.625806,1.983871,8.804839,-0.477646,1.546445,NaN,NaN,NaN,1927-01-01,1927
2,4684,USC00011694,1928,1,40.4,13.132258,-0.312903,6.409677,0.041777,0.696305,NaN,NaN,NaN,1928-01-01,1928
3,4685,USC00011694,1929,1,169.8,15.196774,2.238710,8.717742,1.321482,0.299882,1.508379,NaN,NaN,1929-01-01,1929
4,4686,USC00011694,1930,1,109.8,12.148387,1.009677,6.579032,0.892219,0.796642,1.284593,NaN,NaN,1930-01-01,1930
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1035595,1057166,USW00094967,2025,5,65.1,21.112903,6.529032,13.820968,-0.433724,-0.037004,-0.514225,-1.129750,-1.603309,2025-05-01,2025
1035596,1057167,USW00094967,2025,6,108.2,23.646667,11.856667,17.751667,0.173498,0.096011,-0.232039,-1.462754,-1.274284,2025-06-01,2025
1035597,1057168,USW00094967,2025,7,73.6,26.270968,14.512903,20.391935,-0.149944,-0.352688,-0.234967,-1.530796,-1.072058,2025-07-01,2025
1035598,1057169,USW00094967,2025,8,29.5,25.522581,13.703226,19.612903,-1.219299,-0.725286,-0.661889,-1.687132,-1.002230,2025-08-01,2025


In [3]:
# ============================================================
# SPEI Monthly Trend Analysis (Modified MK + Sen slope)
# - Monthly (NOT annual)
# - Two windows: 1926–2025 (100-yr) and 1996–2025 (30-yr)
# - Summaries by ecoregion + CONUS using Thiessen polygon areas
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import pymannkendall as mk
import data_loader as dl


def get_project_root() -> Path:
    cwd = Path.cwd().resolve()
    return cwd.parent if cwd.name == "scripts" else cwd


def find_project_root(marker: str) -> Path:
    base = get_project_root()
    required_path = base / marker
    if not required_path.exists():
        raise FileNotFoundError(f"Required project file not found: {required_path}")
    return base


def find_github_ready_dir() -> Path:
    return get_project_root()


# ============================================================
# Helpers
# ============================================================
def ensure_month_start(dt: pd.Series) -> pd.Series:
    """
    Ensure datetime and normalize to month start (YYYY-MM-01).
    Works across pandas versions.
    """
    d = pd.to_datetime(dt, errors="coerce")
    return d.dt.to_period("M").dt.start_time

def clip_spei_spi(df: pd.DataFrame, lower_bound: float = -3.0, upper_bound: float = 3.0) -> pd.DataFrame:
    """Clip all SPEI/SPI columns to [lower_bound, upper_bound]."""
    cols = [c for c in df.columns if ("spei" in c.lower()) or ("spi" in c.lower())]
    for c in cols:
        df[c] = pd.to_numeric(df[c], errors="coerce").clip(lower=lower_bound, upper=upper_bound)
    return df


# ============================================================
# Modified Mann–Kendall + Sen's slope
# ============================================================

def modified_mk_test(series, alpha: float = 0.05) -> dict:
    """
    Hamed & Rao Modified Mann–Kendall test.
    Returns trend, p, slope, h, z, tau, s, var_s, intercept.
    """
    s = pd.Series(series).astype(float).dropna()

    # safety: too few points => return NA
    if len(s) < 12:  # at least ~1 year of months (you can raise/lower)
        return {
            "trend": "insufficient_data",
            "p": np.nan,
            "slope": np.nan,
            "h": False,
            "z": np.nan,
            "tau": np.nan,
            "s": np.nan,
            "var_s": np.nan,
            "intercept": np.nan,
            "n": len(s),
        }

    res = mk.hamed_rao_modification_test(s, alpha=alpha)
    return {
        "trend": res.trend,
        "p": res.p,
        "slope": res.slope,
        "h": res.h,
        "z": res.z,
        "tau": res.Tau,
        "s": res.s,
        "var_s": res.var_s,
        "intercept": res.intercept,
        "n": len(s),
    }


def mk_for_dataframe(
    df: pd.DataFrame,
    value_cols,
    group_col: str = "ID",
    time_col: str | None = "DATE",
    alpha: float = 0.05,
) -> pd.DataFrame:
    """
    Apply Modified MK to multiple columns grouped by ID.
    Assumes time-ordered series per group; sorts if time_col is provided.
    """
    if time_col is not None:
        df = df.sort_values([group_col, time_col])

    records = []
    for gid, g in df.groupby(group_col):
        for col in value_cols:
            res = modified_mk_test(g[col].values, alpha=alpha)
            records.append({
                group_col: gid,
                "variable": col,
                "trend": res["trend"],
                "p": res["p"],
                "slope": res["slope"],
                "h": res["h"],
                "z": res["z"],
                "tau": res["tau"],
                "s": res["s"],
                "var_s": res["var_s"],
                "intercept": res["intercept"],
                "n": res["n"],
            })

    return pd.DataFrame.from_records(records)


def mk_two_timeframes_monthly(
    df: pd.DataFrame,
    value_cols,
    group_col: str = "ID",
    time_col: str = "DATE",
    alpha: float = 0.05,
    full_start: str = "1926-01-01",
    full_end: str = "2025-12-01",
    recent_start: str = "1996-01-01",
    recent_end: str = "2025-12-01",
) -> pd.DataFrame:
    """
    Monthly MK in two windows:
      - full: 1926-01 to 2025-12
      - recent: 1996-01 to 2025-12
    """
    d = df.copy()
    d[time_col] = ensure_month_start(d[time_col])

    full = d[(d[time_col] >= full_start) & (d[time_col] <= full_end)].copy()
    recent = d[(d[time_col] >= recent_start) & (d[time_col] <= recent_end)].copy()

    res_full = mk_for_dataframe(full, value_cols=value_cols, group_col=group_col, time_col=time_col, alpha=alpha)
    res_full["window"] = f"{full_start[:4]}-{full_end[:4]}"
    res_full["window_type"] = "full_100yr"

    res_recent = mk_for_dataframe(recent, value_cols=value_cols, group_col=group_col, time_col=time_col, alpha=alpha)
    res_recent["window"] = f"{recent_start[:4]}-{recent_end[:4]}"
    res_recent["window_type"] = "recent_30yr"

    return pd.concat([res_full, res_recent], ignore_index=True)


# ============================================================
# Summaries by ecoregion + CONUS
# ============================================================

def summarize_trends_by_ecoregion(
    mk_df: pd.DataFrame,
    area_df: pd.DataFrame,
    variable: str,
    id_col: str = "ID",
    domain_col: str = "domainName",
    area_col: str = "Area",
    slope_factor: float = 120.0,  # monthly slope -> per decade: (units/month)*120
) -> pd.DataFrame:
    """
    Summarize significant increasing/decreasing trends:
    - PA = % area with significant trend (h=True)
    - slope stats computed over significant stations only

    For monthly data:
      slope_factor=120 converts slope per month -> per decade (10*12 months).
    """
    mk_var = mk_df[mk_df["variable"] == variable].copy()

    merged = mk_var.merge(
        area_df[[id_col, domain_col, area_col]],
        on=id_col,
        how="left",
    )

    records = []
    trend_dirs = ["increasing", "decreasing"]

    # Domain-level
    for (dom, win_type, win), g in merged.groupby([domain_col, "window_type", "window"]):
        total_area = g[area_col].sum()

        for tdir in trend_dirs:
            g_tr = g[(g["h"] == True) & (g["trend"] == tdir)]
            sig_area = g_tr[area_col].sum()
            PA = 100.0 * sig_area / total_area if total_area > 0 else np.nan

            slopes = g_tr["slope"] * slope_factor
            if slopes.empty:
                s_min = s_max = s_mean = s_med = np.nan
            else:
                s_min = slopes.min()
                s_max = slopes.max()
                s_mean = slopes.mean()
                s_med = slopes.median()

            records.append({
                domain_col: dom,
                "window": win,
                "window_type": win_type,
                "trend_dir": tdir,
                "PA": PA,
                "slope_min": s_min,
                "slope_max": s_max,
                "slope_mean": s_mean,
                "slope_median": s_med,
                "n_sig": int(len(g_tr)),
            })

    # CONUS-level
    for (win_type, win), g in merged.groupby(["window_type", "window"]):
        total_area = g[area_col].sum()

        for tdir in trend_dirs:
            g_tr = g[(g["h"] == True) & (g["trend"] == tdir)]
            sig_area = g_tr[area_col].sum()
            PA = 100.0 * sig_area / total_area if total_area > 0 else np.nan

            slopes = g_tr["slope"] * slope_factor
            if slopes.empty:
                s_min = s_max = s_mean = s_med = np.nan
            else:
                s_min = slopes.min()
                s_max = slopes.max()
                s_mean = slopes.mean()
                s_med = slopes.median()

            records.append({
                domain_col: "CONUS",
                "window": win,
                "window_type": win_type,
                "trend_dir": tdir,
                "PA": PA,
                "slope_min": s_min,
                "slope_max": s_max,
                "slope_mean": s_mean,
                "slope_median": s_med,
                "n_sig": int(len(g_tr)),
            })

    return pd.DataFrame.from_records(records)


def make_grouped_pivot(summary_df: pd.DataFrame, domain_col: str = "domainName") -> pd.DataFrame:
    """
    Pivot table where 30-yr columns are grouped together, then 100-yr.
    """
    pivot = summary_df.pivot_table(
        index=domain_col,
        columns=["window_type", "trend_dir"],
        values=["PA", "slope_min", "slope_max", "slope_mean", "slope_median"],
    )

    ordered_cols = [
        ("PA", "recent_30yr", "increasing"),
        ("PA", "recent_30yr", "decreasing"),
        ("slope_min", "recent_30yr", "increasing"),
        ("slope_max", "recent_30yr", "increasing"),
        ("slope_mean", "recent_30yr", "increasing"),
        ("slope_median", "recent_30yr", "increasing"),
        ("slope_min", "recent_30yr", "decreasing"),
        ("slope_max", "recent_30yr", "decreasing"),
        ("slope_mean", "recent_30yr", "decreasing"),
        ("slope_median", "recent_30yr", "decreasing"),

        ("PA", "full_100yr", "increasing"),
        ("PA", "full_100yr", "decreasing"),
        ("slope_min", "full_100yr", "increasing"),
        ("slope_max", "full_100yr", "increasing"),
        ("slope_mean", "full_100yr", "increasing"),
        ("slope_median", "full_100yr", "increasing"),
        ("slope_min", "full_100yr", "decreasing"),
        ("slope_max", "full_100yr", "decreasing"),
        ("slope_mean", "full_100yr", "decreasing"),
        ("slope_median", "full_100yr", "decreasing"),
    ]

    ordered_cols = [c for c in ordered_cols if c in pivot.columns]
    return pivot.reindex(columns=ordered_cols)


def summarize_and_save_for_variable(
    mk_two: pd.DataFrame,
    df_neon_area: pd.DataFrame,
    variable: str,
    out_dir: Path,
    slope_factor: float = 120.0,
) -> None:
    """
    Save summary + pivot for one SPEI variable.
    """
    summary_var = summarize_trends_by_ecoregion(
        mk_df=mk_two,
        area_df=df_neon_area,
        variable=variable,
        slope_factor=slope_factor,
    )

    # Round for clean tables
    summary_var[["PA", "slope_min", "slope_max", "slope_mean", "slope_median"]] = (
        summary_var[["PA", "slope_min", "slope_max", "slope_mean", "slope_median"]].round(4)
    )

    pivot_var = make_grouped_pivot(summary_var)

    summary_path = out_dir / f"summary_{variable}_monthly_by_domain_conus.csv"
    pivot_path = out_dir / f"pivot_{variable}_monthly_for_word_conus.csv"

    summary_var.to_csv(summary_path, index=False)
    pivot_var.to_csv(pivot_path)

    print(f"Saved summary -> {summary_path}")
    print(f"Saved pivot   -> {pivot_path}")


# ============================================================
# MAIN
# ============================================================

def main() -> None:
    out_dir = find_github_ready_dir() / "data_intermediate_spei_monthly"
    out_dir.mkdir(parents=True, exist_ok=True)

    # ---- Load your monthly dataframe ----
    df_monthly = dl.df_monthly_updated.copy()

    # Ensure DATE exists and is month-start
    if "DATE" not in df_monthly.columns:
        df_monthly["DATE"] = pd.to_datetime(df_monthly[["YEAR", "MONTH"]].assign(DAY=1))
    df_monthly["DATE"] = ensure_month_start(df_monthly["DATE"])

    # Clip SPEI
    df_monthly = clip_spei_spi(df_monthly, -3.0, 3.0)

    # ---- Load area + ecoregion lookup ----
    base_dir = find_project_root("data_raw/Thiessen_Polygons_100yrs.csv")
    data_raw_dir = base_dir / "data_raw"

    df_neon = pd.read_csv(data_raw_dir / "Thiessen_Polygons_100yrs.csv")
    df_neon = df_neon[["ID_1", "Area_sqkm"]]
    df_neon.columns = ["ID", "Area"]

    df_id_neon = pd.read_csv(data_raw_dir / "ID_ecoregion.csv")
    df_id_neon_updated = df_id_neon.copy()

    replace_map = {
        "Atlantic Neotropical": "Southeast",
        "Appalachians / Cumberland Plateau": "Appalachians",
        "Southern Rockies / Colorado Plateau": "Colorado Plateau",
    }
    df_id_neon_updated["domainName"] = df_id_neon_updated["domainName"].replace(replace_map)
    df_id_neon_updated = df_id_neon_updated[["ID", "domainName"]]

    df_neon_area = df_neon.merge(df_id_neon_updated, on="ID", how="inner")

    # ---- SPEI monthly vars ----
    spei_cols = ["spei1", "spei3", "spei6", "spei12"]
    spei_cols = [c for c in spei_cols if c in df_monthly.columns]
    if not spei_cols:
        raise ValueError("None of spei1/spei3/spei6/spei12 found in df_monthly.")

    # ---- Run MK for two timeframes (MONTHLY) ----
    mk_two = mk_two_timeframes_monthly(
        df_monthly,
        value_cols=spei_cols,
        group_col="ID",
        time_col="DATE",
        alpha=0.05,
        full_start="1926-01-01",
        full_end="2025-12-01",
        recent_start="1996-01-01",
        recent_end="2025-12-01",
    )

    mk_out = out_dir / "mk_two_spei_monthly.csv"
    mk_two.to_csv(mk_out, index=False)
    print(f"\nSaved MK results (SPEI monthly) -> {mk_out}")

    # ---- Summaries for each SPEI variable ----
    # slope_factor=120 converts slope/month -> slope/decade
    for var in spei_cols:
        summarize_and_save_for_variable(
            mk_two=mk_two,
            df_neon_area=df_neon_area,
            variable=var,
            out_dir=out_dir,
            slope_factor=120.0,
        )

    print("\nAll SPEI monthly variables processed.")


if __name__ == "__main__":
    main()


C:\Users\adeba\anaconda3\envs\advance\Lib\site-packages\pymannkendall\pymannkendall.py:99: RuntimeWarning: invalid value encountered in sqrt
  z = (s - 1)/np.sqrt(var_s)
C:\Users\adeba\anaconda3\envs\advance\Lib\site-packages\pymannkendall\pymannkendall.py:103: RuntimeWarning: invalid value encountered in sqrt
  z = (s + 1)/np.sqrt(var_s)
C:\Users\adeba\anaconda3\envs\advance\Lib\site-packages\pymannkendall\pymannkendall.py:103: RuntimeWarning: invalid value encountered in sqrt
  z = (s + 1)/np.sqrt(var_s)
C:\Users\adeba\anaconda3\envs\advance\Lib\site-packages\pymannkendall\pymannkendall.py:103: RuntimeWarning: invalid value encountered in sqrt
  z = (s + 1)/np.sqrt(var_s)
C:\Users\adeba\anaconda3\envs\advance\Lib\site-packages\pymannkendall\pymannkendall.py:99: RuntimeWarning: invalid value encountered in sqrt
  z = (s - 1)/np.sqrt(var_s)
C:\Users\adeba\anaconda3\envs\advance\Lib\site-packages\pymannkendall\pymannkendall.py:99: RuntimeWarning: invalid value encountered in sqrt
  z = 


Saved MK results (SPEI monthly) -> C:\Users\adeba\OneDrive\Documents\Datascience\Full_climate_project\climate_100yr\climate_100yr_github_ready\data_intermediate_spei_monthly\mk_two_spei_monthly.csv
Saved summary -> C:\Users\adeba\OneDrive\Documents\Datascience\Full_climate_project\climate_100yr\climate_100yr_github_ready\data_intermediate_spei_monthly\summary_spei1_monthly_by_domain_conus.csv
Saved pivot   -> C:\Users\adeba\OneDrive\Documents\Datascience\Full_climate_project\climate_100yr\climate_100yr_github_ready\data_intermediate_spei_monthly\pivot_spei1_monthly_for_word_conus.csv
Saved summary -> C:\Users\adeba\OneDrive\Documents\Datascience\Full_climate_project\climate_100yr\climate_100yr_github_ready\data_intermediate_spei_monthly\summary_spei3_monthly_by_domain_conus.csv
Saved pivot   -> C:\Users\adeba\OneDrive\Documents\Datascience\Full_climate_project\climate_100yr\climate_100yr_github_ready\data_intermediate_spei_monthly\pivot_spei3_monthly_for_word_conus.csv
Saved summary -

In [4]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
SPEI Drought-Months Summary (Monthly, Station-Based, CONUS + Ecoregions)

Goal
----
For each station and each SPEI variable (spei1, spei3, spei6, spei12),
compute the number of drought months (SPEI < -1) normalized to "months per decade"
for three periods:
  1) Centennial: 1926–2025
  2) Recent:     1996–2025
  3) Penultimate:1966–1995

Then compute differences:
  - recent_minus_centennial
  - recent_minus_penultimate

Summarize min/max/mean/median by ecoregion and for CONUS.

Inputs expected
---------------
1) data_loader.py provides:
   - dl.df_monthly_updated (monthly dataframe with columns:
        ID, DATE (or YEAR & MONTH), spei1, spei3, spei6, spei12, ...)

2) data_raw/ID_ecoregion.csv with columns:
   - ID, domainName (plus other columns ok)

Outputs
-------
Creates folder: climate_100yr_github_ready/data_intermediate_spei_drought/
For each speiX:
  - speiX_station_drought_months.csv
  - speiX_drought_summary.csv
"""

from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd
import data_loader as dl


def get_project_root() -> Path:
    cwd = Path.cwd().resolve()
    return cwd.parent if cwd.name == "scripts" else cwd


def find_project_root(marker: str) -> Path:
    base = get_project_root()
    required_path = base / marker
    if not required_path.exists():
        raise FileNotFoundError(f"Required project file not found: {required_path}")
    return base


def find_github_ready_dir() -> Path:
    return get_project_root()


# ============================================================
# Helpers
# ============================================================

def clip_spei_spi(df: pd.DataFrame, lower: float = -3.0, upper: float = 3.0) -> pd.DataFrame:
    """Clip all columns containing 'spei' or 'spi' (case-insensitive) to [lower, upper]."""
    out = df.copy()
    cols = [c for c in out.columns if ("spei" in c.lower()) or ("spi" in c.lower())]
    for c in cols:
        out[c] = pd.to_numeric(out[c], errors="coerce").clip(lower=lower, upper=upper)
    return out


def ensure_date_column(df: pd.DataFrame, date_col: str = "DATE") -> pd.DataFrame:
    """
    Ensure df has a proper datetime DATE column.
    If DATE is missing, builds DATE from YEAR and MONTH (day=1).
    """
    out = df.copy()
    if date_col not in out.columns:
        if {"YEAR", "MONTH"}.issubset(out.columns):
            out[date_col] = pd.to_datetime(out[["YEAR", "MONTH"]].assign(DAY=1), errors="coerce")
        else:
            raise ValueError("DATE column missing and YEAR/MONTH not available to construct DATE.")
    else:
        out[date_col] = pd.to_datetime(out[date_col], errors="coerce")
    return out


def drought_months_per_decade(
    df: pd.DataFrame,
    spei_col: str,
    id_col: str = "ID",
    date_col: str = "DATE",
    start_year: int = 1926,
    end_year: int = 2025,
    threshold: float = -1.0,
    min_months_required: int = 12,
) -> pd.DataFrame:
    """
    For each station, count drought months (SPEI < threshold) in [start_year, end_year],
    then normalize to drought months per decade.

    months_per_decade = (drought_months / n_years) * 10

    If a station has fewer than `min_months_required` valid months in the window,
    result is NaN for that station (to avoid unstable rates).
    """
    d = df.copy()
    d["YEAR_"] = d[date_col].dt.year

    win = d[(d["YEAR_"] >= start_year) & (d["YEAR_"] <= end_year)].copy()
    win[spei_col] = pd.to_numeric(win[spei_col], errors="coerce")

    # Count valid months and drought months per station
    valid_counts = win.groupby(id_col)[spei_col].apply(lambda x: x.notna().sum()).reset_index(name="n_valid_months")
    drought_counts = (
        win.assign(is_drought=win[spei_col] < threshold)
        .groupby(id_col)["is_drought"]
        .sum()
        .reset_index(name="drought_months")
    )

    out = valid_counts.merge(drought_counts, on=id_col, how="outer")

    n_years = end_year - start_year + 1
    out["drought_months_per_decade"] = out["drought_months"] / n_years * 10.0

    # Enforce minimum data requirement
    out.loc[out["n_valid_months"] < min_months_required, "drought_months_per_decade"] = np.nan

    return out[[id_col, "drought_months_per_decade"]]


def compute_spei_drought_metrics(
    df_monthly: pd.DataFrame,
    spei_col: str,
    id_col: str = "ID",
    date_col: str = "DATE",
    threshold: float = -1.0,
) -> pd.DataFrame:
    """
    Compute station-level drought months per decade for:
      - centennial (1926–2025)
      - recent (1996–2025)
      - penultimate (1966–1995)
    and differences:
      - recent_minus_centennial
      - recent_minus_penultimate
    """
    cent = drought_months_per_decade(
        df_monthly, spei_col, id_col, date_col, 1926, 2025, threshold=threshold
    ).rename(columns={"drought_months_per_decade": f"{spei_col}_centennial"})

    recent = drought_months_per_decade(
        df_monthly, spei_col, id_col, date_col, 1996, 2025, threshold=threshold
    ).rename(columns={"drought_months_per_decade": f"{spei_col}_recent"})

    penult = drought_months_per_decade(
        df_monthly, spei_col, id_col, date_col, 1966, 1995, threshold=threshold
    ).rename(columns={"drought_months_per_decade": f"{spei_col}_penultimate"})

    out = cent.merge(recent, on=id_col, how="outer").merge(penult, on=id_col, how="outer")

    out[f"{spei_col}_recent_minus_centennial"] = out[f"{spei_col}_recent"] - out[f"{spei_col}_centennial"]
    out[f"{spei_col}_recent_minus_penultimate"] = out[f"{spei_col}_recent"] - out[f"{spei_col}_penultimate"]

    return out


def attach_ecoregions(
    station_df: pd.DataFrame,
    domains_df: pd.DataFrame,
    id_col: str = "ID",
    domain_col: str = "domainName",
) -> pd.DataFrame:
    """Attach domainName (ecoregion) to station-level dataframe."""
    return station_df.merge(domains_df[[id_col, domain_col]], on=id_col, how="left")


def summarize_by_region_and_conus(
    df: pd.DataFrame,
    metrics: list[str],
    domain_col: str = "domainName",
) -> pd.DataFrame:
    """
    Summarize min/max/mean/median of metrics by ecoregion and for CONUS.
    """
    agg = {m: ["min", "max", "mean", "median"] for m in metrics}
    region = df.groupby(domain_col).agg(agg)

    region.columns = [f"{m}_{s}" for m, s in region.columns]
    region = region.reset_index()

    conus_vals = df[metrics].agg(["min", "max", "mean", "median"]).T
    conus_row = {domain_col: "CONUS"}
    for m in metrics:
        conus_row[f"{m}_min"] = conus_vals.loc[m, "min"]
        conus_row[f"{m}_max"] = conus_vals.loc[m, "max"]
        conus_row[f"{m}_mean"] = conus_vals.loc[m, "mean"]
        conus_row[f"{m}_median"] = conus_vals.loc[m, "median"]

    return pd.concat([region, pd.DataFrame([conus_row])], ignore_index=True)


# ============================================================
# Main
# ============================================================

def main() -> None:
    out_dir = find_github_ready_dir() / "data_intermediate_spei_drought"
    out_dir.mkdir(parents=True, exist_ok=True)

    # --- Load monthly data ---
    df_monthly = dl.df_monthly_updated.copy()
    df_monthly = ensure_date_column(df_monthly, date_col="DATE")
    df_monthly = clip_spei_spi(df_monthly, lower=-3.0, upper=3.0)

    # Optional: keep only target analysis years (not required, but keeps things tidy)
    df_monthly = df_monthly[(df_monthly["DATE"] >= "1926-01-01") & (df_monthly["DATE"] <= "2025-12-31")].copy()

    # --- Load ecoregions / domains ---
    base_dir = find_project_root("data_raw/ID_ecoregion.csv")
    domains_path = base_dir / "data_raw" / "ID_ecoregion.csv"
    df_domains = pd.read_csv(domains_path)

    # Normalize names (your mapping)
    replace_map = {
        "Atlantic Neotropical": "Southeast",
        "Appalachians / Cumberland Plateau": "Appalachians",
        "Southern Rockies / Colorado Plateau": "Colorado Plateau",
    }
    df_domains["domainName"] = df_domains["domainName"].replace(replace_map)
    df_domains = df_domains[["ID", "domainName"]].drop_duplicates()

    # --- SPEI variables to process ---
    spei_vars = ["spei1", "spei3", "spei6", "spei12"]
    spei_vars = [v for v in spei_vars if v in df_monthly.columns]
    if not spei_vars:
        raise ValueError("No SPEI columns found. Expected one of: spei1, spei3, spei6, spei12")

    # Drought definition
    drought_threshold = -1.0  # SPEI < -1

    for spei in spei_vars:
        print(f"\n=== Processing {spei}: drought months (SPEI < {drought_threshold}) per decade ===")

        # Station-level drought months per decade (3 periods + deltas)
        station = compute_spei_drought_metrics(
            df_monthly=df_monthly,
            spei_col=spei,
            id_col="ID",
            date_col="DATE",
            threshold=drought_threshold,
        )

        station = attach_ecoregions(station, df_domains, id_col="ID", domain_col="domainName")

        metrics = [
            f"{spei}_centennial",
            f"{spei}_recent",
            f"{spei}_penultimate",
            f"{spei}_recent_minus_centennial",
            f"{spei}_recent_minus_penultimate",
        ]

        summary = summarize_by_region_and_conus(
            station,
            metrics=metrics,
            domain_col="domainName",
        )

        # Round summaries for nice tables (station-level left as-is)
        summary = summary.round(3)

        station_out = out_dir / f"{spei}_station_drought_months_per_decade.csv"
        summary_out = out_dir / f"{spei}_drought_months_per_decade_summary.csv"

        station.to_csv(station_out, index=False)
        summary.to_csv(summary_out, index=False)

        print(f"Saved station-level -> {station_out}")
        print(f"Saved summary       -> {summary_out}")

    print(f"\nDone. Outputs saved in: {out_dir.resolve()}")


if __name__ == "__main__":
    main()



=== Processing spei1: drought months (SPEI < -1.0) per decade ===
Saved station-level -> C:\Users\adeba\OneDrive\Documents\Datascience\Full_climate_project\climate_100yr\climate_100yr_github_ready\data_intermediate_spei_drought\spei1_station_drought_months_per_decade.csv
Saved summary       -> C:\Users\adeba\OneDrive\Documents\Datascience\Full_climate_project\climate_100yr\climate_100yr_github_ready\data_intermediate_spei_drought\spei1_drought_months_per_decade_summary.csv

=== Processing spei3: drought months (SPEI < -1.0) per decade ===
Saved station-level -> C:\Users\adeba\OneDrive\Documents\Datascience\Full_climate_project\climate_100yr\climate_100yr_github_ready\data_intermediate_spei_drought\spei3_station_drought_months_per_decade.csv
Saved summary       -> C:\Users\adeba\OneDrive\Documents\Datascience\Full_climate_project\climate_100yr\climate_100yr_github_ready\data_intermediate_spei_drought\spei3_drought_months_per_decade_summary.csv

=== Processing spei6: drought months (SPE